# Load data

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA_PATH = Path(
    "data/synthetic/"
    "gta_store_product_week_history_v1.parquet"
)

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

df = pd.read_parquet(DATA_PATH)

df["week_start"] = pd.to_datetime(
    df["week_start"]
)

print(df.shape)
print(
    df["week_start"].min(),
    "→",
    df["week_start"].max()
)

print(
    "Promotion rate:",
    round(
        df["promotion"].mean() * 100,
        2
    ),
    "%"
)

(468000, 54)
2024-09-16 00:00:00 → 2026-09-07 00:00:00
Promotion rate: 14.78 %


# Create past-sales features

In [2]:
df = (
    df
    .sort_values(
        [
            "store_id",
            "product_id",
            "week_start"
        ]
    )
    .reset_index(drop=True)
)

group = df.groupby(
    ["store_id", "product_id"]
)["units_sold"]

df["lag_1"] = group.shift(1)
df["lag_4"] = group.shift(4)

df["rolling_mean_4"] = (
    group.transform(
        lambda x:
        x.shift(1)
        .rolling(
            4,
            min_periods=1
        )
        .mean()
    )
)

df["rolling_mean_8"] = (
    group.transform(
        lambda x:
        x.shift(1)
        .rolling(
            8,
            min_periods=1
        )
        .mean()
    )
)

df["month"] = (
    df["week_start"].dt.month
)

df["week_of_year"] = (
    df["week_start"]
    .dt.isocalendar()
    .week
    .astype(int)
)

df = df[
    df["lag_1"].notna()
].copy()

print(df.shape)

(463500, 59)


# Select PRE-TREATMENT features

In [3]:
categorical_features = [
    "store_id",
    "product_id",
    "category",
    "city"
]

numeric_features = [
    "regular_price",
    "standard_discount_pct",

    "inventory",

    "lag_1",
    "lag_4",
    "rolling_mean_4",
    "rolling_mean_8",

    "mean_temperature",
    "total_rain",
    "total_snow",

    "month",
    "week_of_year",

    "population_5km",
    "median_household_income_5km",

    "offices",
    "commercial",
    "universities",
    "colleges"
]

FEATURES = (
    categorical_features
    + numeric_features
)

OUTCOME = "units_sold"
TREATMENT = "promotion"

# Time split

In [4]:
weeks = np.array(
    sorted(
        df["week_start"].unique()
    )
)

val_start = weeks[-24]
test_start = weeks[-12]

train = df[
    df["week_start"] < val_start
].copy()

val = df[
    (df["week_start"] >= val_start)
    & (df["week_start"] < test_start)
].copy()

test = df[
    df["week_start"] >= test_start
].copy()

print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

print(
    "Train promotion rate:",
    train["promotion"].mean()
)

print(
    "Validation promotion rate:",
    val["promotion"].mean()
)

print(
    "Test promotion rate:",
    test["promotion"].mean()
)

Train: (355500, 59)
Validation: (54000, 59)
Test: (54000, 59)
Train promotion rate: 0.14742334739803095
Validation promotion rate: 0.1496111111111111
Test promotion rate: 0.14909259259259258


# Split treatment/control

In [5]:
train_control = train[
    train["promotion"] == 0
].copy()

train_treated = train[
    train["promotion"] == 1
].copy()

print(
    "Control:",
    len(train_control)
)

print(
    "Treated:",
    len(train_treated)
)

Control: 303091
Treated: 52409


In [6]:
X_control = (
    train_control[FEATURES]
    .copy()
)

y_control = (
    train_control[OUTCOME]
    .copy()
)

X_treated = (
    train_treated[FEATURES]
    .copy()
)

y_treated = (
    train_treated[OUTCOME]
    .copy()
)

X_val = val[FEATURES].copy()
X_test = test[FEATURES].copy()

In [7]:
for col in categorical_features:

    X_control[col] = (
        X_control[col]
        .fillna("Unknown")
        .astype(str)
    )

    X_treated[col] = (
        X_treated[col]
        .fillna("Unknown")
        .astype(str)
    )

    X_val[col] = (
        X_val[col]
        .fillna("Unknown")
        .astype(str)
    )

    X_test[col] = (
        X_test[col]
        .fillna("Unknown")
        .astype(str)
    )

# Train CONTROL model

In [8]:
model_control = CatBoostRegressor(
    iterations=700,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

model_control.fit(
    X_control,
    y_control,
    cat_features=categorical_features
)

0:	learn: 6.6522110	total: 108ms	remaining: 1m 15s
100:	learn: 3.9278443	total: 3.18s	remaining: 18.8s
200:	learn: 3.8935011	total: 6.73s	remaining: 16.7s
300:	learn: 3.8648512	total: 10.3s	remaining: 13.7s
400:	learn: 3.8437778	total: 14.1s	remaining: 10.5s
500:	learn: 3.8248558	total: 17.9s	remaining: 7.09s
600:	learn: 3.8123714	total: 21.8s	remaining: 3.58s
699:	learn: 3.7990842	total: 25.7s	remaining: 0us


CatBoostRegressor(depth=8, iterations=700, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

# Train TREATMENT model

In [9]:
model_treated = CatBoostRegressor(
    iterations=700,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

model_treated.fit(
    X_treated,
    y_treated,
    cat_features=categorical_features
)

0:	learn: 7.7311629	total: 10.5ms	remaining: 7.34s
100:	learn: 3.8663640	total: 764ms	remaining: 4.53s
200:	learn: 3.7951245	total: 1.57s	remaining: 3.89s
300:	learn: 3.7310166	total: 2.41s	remaining: 3.19s
400:	learn: 3.6643410	total: 3.31s	remaining: 2.47s
500:	learn: 3.6108477	total: 4.18s	remaining: 1.66s
600:	learn: 3.5569349	total: 5.05s	remaining: 832ms
699:	learn: 3.5100837	total: 5.92s	remaining: 0us


CatBoostRegressor(depth=8, iterations=700, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

# Predict both worlds

In [10]:
val_pred_control = np.clip(
    model_control.predict(X_val),
    0,
    None
)

val_pred_treated = np.clip(
    model_treated.predict(X_val),
    0,
    None
)

val_uplift_pred = (
    val_pred_treated
    - val_pred_control
)

In [11]:
val_uplift_pred

array([1.73121249, 3.11980127, 1.93728463, ..., 5.1325169 , 2.95077659,
       3.25199644], shape=(54000,))

# Compare against TRUE uplift

In [12]:
val_true_uplift = (
    val["true_incremental_units"]
    .to_numpy()
)

uplift_mae = mean_absolute_error(
    val_true_uplift,
    val_uplift_pred
)

uplift_rmse = np.sqrt(
    mean_squared_error(
        val_true_uplift,
        val_uplift_pred
    )
)

uplift_corr = np.corrcoef(
    val_true_uplift,
    val_uplift_pred
)[0, 1]

print(
    "Uplift MAE:",
    round(uplift_mae, 4)
)

print(
    "Uplift RMSE:",
    round(uplift_rmse, 4)
)

print(
    "Uplift correlation:",
    round(uplift_corr, 4)
)

Uplift MAE: 0.8579
Uplift RMSE: 1.1748
Uplift correlation: 0.8303


# Compare average treatment effect

In [13]:
print(
    "TRUE average uplift:",
    round(
        val_true_uplift.mean(),
        3
    )
)

print(
    "PREDICTED average uplift:",
    round(
        val_uplift_pred.mean(),
        3
    )
)

TRUE average uplift: 2.848
PREDICTED average uplift: 2.209


# Create validation result table

In [14]:
val_results = val[
    [
        "week_start",
        "store_id",
        "store_name",
        "product_id",
        "product_name",
        "category",

        "promotion",

        "units_sold",

        "true_uplift_rate",
        "true_incremental_units",
        "true_incremental_profit"
    ]
].copy()

val_results[
    "pred_units_control"
] = val_pred_control

val_results[
    "pred_units_treated"
] = val_pred_treated

val_results[
    "predicted_uplift_units"
] = val_uplift_pred

val_results[
    "uplift_error"
] = (
    val_results[
        "predicted_uplift_units"
    ]
    - val_results[
        "true_incremental_units"
    ]
)

display(
    val_results.head(20)
)

,week_start,store_id,store_name,product_id,product_name,category,promotion,units_sold,true_uplift_rate,true_incremental_units,true_incremental_profit,pred_units_control,pred_units_treated,predicted_uplift_units,uplift_error
80,2026-03-30,1004,Toronto Stockyards,SYN_0001,Hot Beverages Product 01,Hot Beverages,0,25,0.09,1.698569,-0.504719,19.681239,21.412451,1.731212,0.032644
81,2026-04-06,1004,Toronto Stockyards,SYN_0001,Hot Beverages Product 01,Hot Beverages,0,18,0.19,3.585868,6.906184,20.425734,23.545536,3.119801,-0.466066
82,2026-04-13,1004,Toronto Stockyards,SYN_0001,Hot Beverages Product 01,Hot Beverages,0,25,0.09,1.698569,-0.504719,19.800667,21.737952,1.937285,0.238716
83,2026-04-20,1004,Toronto Stockyards,SYN_0001,Hot Beverages Product 01,Hot Beverages,0,19,0.09,1.698569,-0.504719,18.887063,19.249050,0.361987,-1.336582
84,2026-04-27,1004,Toronto Stockyards,SYN_0001,Hot Beverages Product 01,Hot Beverages,0,23,0.09,1.698569,-0.504719,19.837575,21.951219,2.113643,0.415075
85,2026-05-04,1004,Toronto Stockyards,SYN_0001,Hot Beverages Product 01,Hot Beverages,0,13,0.09,1.698569,-0.504719,19.773134,21.209601,1.436467,-0.262102
86,2026-05-11,1004,Toronto Stockyards,SYN_0001,Hot Beverages Product 01,Hot Beverages,0,17,0.09,1.698569,-0.504719,19.595148,22.316555,2.721407,1.022838
87,2026-05-18,1004,Toronto Stockyards,SYN_0001,Hot Beverages Product 01,Hot Beverages,0,21,0.09,1.698569,-0.504719,19.939000,22.068907,2.129907,0.431339
88,2026-05-25,1004,Toronto Stockyards,SYN_0001,Hot Beverages Product 01,Hot Beverages,1,20,0.09,1.698569,-0.504719,19.589667,21.468233,1.878566,0.179997
89,2026-06-01,1004,Toronto Stockyards,SYN_0001,Hot Beverages Product 01,Hot Beverages,0,21,0.09,1.698569,-0.504719,20.116276,22.436406,2.320130,0.621561


# Uplift@K

In [15]:
ranked = (
    val_results
    .sort_values(
        "predicted_uplift_units",
        ascending=False
    )
    .reset_index(drop=True)
)

In [16]:
def uplift_at_k(
    data,
    pct
):

    n = int(
        len(data) * pct
    )

    top = data.head(n)

    return (
        top[
            "true_incremental_units"
        ].mean()
    )

In [17]:
for pct in [
    0.10,
    0.20,
    0.30
]:

    score = uplift_at_k(
        ranked,
        pct
    )

    print(
        f"Uplift@{int(pct*100)}%:",
        round(score, 3)
    )

Uplift@10%: 5.889
Uplift@20%: 5.142
Uplift@30%: 4.697


# Compare top vs overall

In [18]:
overall_uplift = (
    val_results[
        "true_incremental_units"
    ].mean()
)

top10_uplift = (
    ranked
    .head(
        int(
            len(ranked) * 0.10
        )
    )[
        "true_incremental_units"
    ]
    .mean()
)

print(
    "Overall true uplift:",
    round(
        overall_uplift,
        3
    )
)

print(
    "Top 10% true uplift:",
    round(
        top10_uplift,
        3
    )
)

Overall true uplift: 2.848
Top 10% true uplift: 5.889


# Check store variation

In [19]:
store_uplift = (
    val_results
    .groupby(
        [
            "store_id",
            "store_name"
        ]
    )
    .agg(
        predicted_uplift=(
            "predicted_uplift_units",
            "mean"
        ),

        true_uplift=(
            "true_incremental_units",
            "mean"
        )
    )
    .reset_index()
)

display(
    store_uplift.sort_values(
        "predicted_uplift",
        ascending=False
    )
)

,store_id,store_name,predicted_uplift,true_uplift
11,3106,Toronto Dufferin Mall,3.217050,4.102523
5,1150,Toronto Downtown East,2.721582,3.620994
0,1004,Toronto Stockyards,2.588830,3.387680
10,3105,Toronto North Park,2.480413,3.157344
3,1126,Mississauga East,2.365369,2.962202
9,3055,Mississauga Square One,2.259020,2.971939
4,1139,Toronto Downsview,2.067556,2.704366
12,3130,Brampton North,2.064671,2.711753
1,1061,Mississauga Heartland,1.992337,2.688954
6,1188,Brampton South,1.974201,2.536213


# Check category variation

In [20]:
category_uplift = (
    val_results
    .groupby("category")
    .agg(
        predicted_uplift=(
            "predicted_uplift_units",
            "mean"
        ),

        true_uplift=(
            "true_incremental_units",
            "mean"
        )
    )
    .reset_index()
)

display(
    category_uplift.sort_values(
        "predicted_uplift",
        ascending=False
    )
)

,category,predicted_uplift,true_uplift
3,Dairy,3.041015,3.861996
2,Cold Beverages,2.979720,3.738778
9,Snacks,2.831246,3.603465
0,Breakfast,2.487110,3.295092
6,Household Staples,2.327850,3.012113
8,Ready-to-Eat,1.953488,2.535020
5,Hot Beverages,1.911191,2.283362
1,Cleaning,1.750806,2.275353
4,Frozen Food,1.733224,2.313426
7,Personal Care,1.075671,1.563438


# Final TEST evaluation

In [21]:
test_pred_control = np.clip(
    model_control.predict(
        X_test
    ),
    0,
    None
)

test_pred_treated = np.clip(
    model_treated.predict(
        X_test
    ),
    0,
    None
)

test_uplift_pred = (
    test_pred_treated
    - test_pred_control
)

test_true_uplift = (
    test[
        "true_incremental_units"
    ].to_numpy()
)

test_uplift_mae = mean_absolute_error(
    test_true_uplift,
    test_uplift_pred
)

test_uplift_rmse = np.sqrt(
    mean_squared_error(
        test_true_uplift,
        test_uplift_pred
    )
)

test_uplift_corr = np.corrcoef(
    test_true_uplift,
    test_uplift_pred
)[0, 1]

print(
    "TEST uplift MAE:",
    round(
        test_uplift_mae,
        4
    )
)

print(
    "TEST uplift RMSE:",
    round(
        test_uplift_rmse,
        4
    )
)

print(
    "TEST uplift correlation:",
    round(
        test_uplift_corr,
        4
    )
)

TEST uplift MAE: 0.9568
TEST uplift RMSE: 1.3452
TEST uplift correlation: 0.8283


# Save test predictions

In [22]:
test_results = test[
    [
        "week_start",
        "store_id",
        "store_name",
        "product_id",
        "product_name",
        "category",

        "promotion",
        "units_sold",

        "regular_price",
        "standard_discount_pct",

        "true_incremental_units",
        "true_incremental_profit"
    ]
].copy()

test_results[
    "pred_units_control"
] = test_pred_control

test_results[
    "pred_units_treated"
] = test_pred_treated

test_results[
    "predicted_uplift_units"
] = test_uplift_pred

OUTPUT_PATH = (
    RESULTS_DIR /
    "promotion_uplift_tlearner_v1.parquet"
)

test_results.to_parquet(
    OUTPUT_PATH,
    index=False
)

print(
    "Saved:",
    OUTPUT_PATH
)

Saved: results/promotion_uplift_tlearner_v1.parquet
